# E-commerce Signals from Shopee

Use daily Shopee transaction counts for the consumer-goods category as a leading indicator for the listed names that sell into it — MSN, VNM, MWG. Then fit a simple nowcasting regression for monthly revenue.

**Why this matters**: alt-data lead times are everything. Quarterly earnings land 45 days after period-end; daily marketplace data lands T+1. If the signal correlates with reported revenue, you have a 6-week head start on the consensus.

## Setup

All data here is mocked. Production calls would be:

- `dc.dataset('altdata.shopee.daily_txn_count').to_pandas(category='consumer_goods')`
- `dc.dataset('equity.daily').to_pandas(tickers=['MSN','VNM','MWG'])`
- `dc.dataset('equity.fundamentals.quarterly_revenue').to_pandas(...)`

In [ ]:
import pandas as pd  # noqa
import numpy as np  # noqa
import matplotlib.pyplot as plt

rng = np.random.default_rng(9)
plt.rcParams['figure.figsize'] = (11, 5)

## Step 1 — Pull Shopee daily transaction counts (mocked)

Realistic shape: weekly seasonality (weekend spikes), monthly payday bump, secular growth ~30% YoY.

In [ ]:
dates = pd.date_range('2022-01-01', '2025-04-30', freq='D')
n = len(dates)

trend = np.linspace(1.0, 2.2, n)            # ~30% YoY growth
weekly = 1 + 0.15 * np.sin(2 * np.pi * dates.dayofweek / 7)
monthly = 1 + 0.10 * (dates.day >= 25)       # payday bump
noise = rng.normal(1, 0.05, n)

shopee = pd.Series(
    (8e5 * trend * weekly * monthly * noise).round().astype(int),
    index=dates, name='shopee_txn_consumer_goods',
)
shopee.tail()

## Step 2 — Pull stock prices (mocked)

MSN, VNM, MWG — three names with real consumer-goods exposure. Mocked closes in tens of thousands of VND.

In [ ]:
trading_days = pd.bdate_range('2022-01-01', '2025-04-30')

def mock_prices(start, mu, sigma):
    r = rng.normal(mu, sigma, len(trading_days))
    return pd.Series(start * np.exp(np.cumsum(r)), index=trading_days)

prices = pd.DataFrame({
    'MSN': mock_prices(80_000, 0.0003, 0.018),
    'VNM': mock_prices(75_000, 0.0001, 0.014),
    'MWG': mock_prices(45_000, 0.0008, 0.022),
}).round(0)
prices.tail()

## Step 3 — Smooth the alt-data signal

Raw daily transaction counts are too noisy to compare against weekly stock moves. A 28-day rolling sum captures the underlying demand pulse.

In [ ]:
signal = shopee.rolling(28).sum().dropna().rename('shopee_28d_txn')

fig, ax = plt.subplots()
signal.plot(ax=ax, lw=1.5, color='orange', label='Shopee 28d txn count (consumer goods)')
ax.set_title('Shopee transaction pulse')
ax.set_ylabel('28-day rolling txn count')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Step 4 — Correlate YoY signal with stock returns

Convert the rolling signal to YoY change (strips out seasonality) and compare to trailing 1-month stock returns.

In [ ]:
signal_yoy = signal.pct_change(365).rename('shopee_yoy')
ret_1m = prices.pct_change(21)

merged = pd.concat([signal_yoy, ret_1m], axis=1).dropna()
corr = merged.corr().loc['shopee_yoy', ['MSN', 'VNM', 'MWG']]
print('Correlation: Shopee YoY signal vs 1M stock return\n')
print(corr.round(3))

## Step 5 — Scatter of signal vs returns

Visualize for each name. A clean upward-sloping cloud is what you hope for.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, t in zip(axes, ['MSN', 'VNM', 'MWG']):
    ax.scatter(merged['shopee_yoy'], merged[t], s=8, alpha=0.4)
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('Shopee YoY')
    ax.set_title(f'{t}  (r = {corr[t]:+.2f})')
axes[0].set_ylabel('1M stock return')
plt.tight_layout()
plt.show()

## Step 6 — Nowcast monthly revenue

Build a tiny OLS that maps the alt-data signal to next-quarter reported revenue. In production you'd cross-validate, here we just show the mechanics.

In [ ]:
# Mock quarterly revenue for MSN (trillion VND), correlated with the signal
q_idx = pd.date_range('2022-03-31', '2025-03-31', freq='QE')
shopee_q = signal.resample('QE').last().reindex(q_idx)

true_revenue = (
    18 + 6 * (shopee_q / shopee_q.iloc[0])
    + rng.normal(0, 1.2, len(q_idx))
).round(2)
true_revenue = pd.Series(true_revenue.values, index=q_idx, name='MSN_revenue_T_VND')

# Simple OLS via numpy
x = (shopee_q.values - shopee_q.mean()) / shopee_q.std()
y = true_revenue.values
beta = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
alpha_ = y.mean() - beta * x.mean()
yhat = alpha_ + beta * x

r2 = 1 - ((y - yhat) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f'MSN revenue ~ {alpha_:.2f} + {beta:.2f} * z(shopee_signal)')
print(f'R² = {r2:.3f}')

## Step 7 — Plot actuals vs nowcast

If the model has real signal, the next-period nowcast for the upcoming quarter is your edge — you know revenue before MSN reports it.

In [ ]:
fig, ax = plt.subplots()
ax.plot(q_idx, y, marker='o', lw=2, label='Reported revenue')
ax.plot(q_idx, yhat, marker='s', lw=2, linestyle='--', label='Shopee-based nowcast')
ax.set_ylabel('MSN revenue (T VND)')
ax.set_title(f'MSN quarterly revenue: actuals vs nowcast (R² = {r2:.2f})')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Next steps

- `02-satellite-port-traffic.ipynb` — another lead indicator (Saigon Port)
- `03-social-sentiment.ipynb` — sentiment as a complement to txn volume
- `02-equity-research/01-vn30-fundamentals.ipynb` — combine alt-data score with quality screen